In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Využití pracovní zátěže

<span id="usage"></span>
Využití je měřítkem množství času, po který je QPU uzamčena pro tvou pracovní zátěž, a počítá se různě v závislosti na tom, který režim spuštění používáš.

* Využití Session je wall-clock čas, po který je session aktivní. Více informací o přechodech stavů session najdeš v části [Délka session](/guides/run-jobs-session#session-length).
* Využití Batch je součet kvantového času (čas, který QPU komplex stráví zpracováváním tvé úlohy) všech úloh v dávce.
* Využití jedné úlohy je kvantový čas, který úloha spotřebuje při zpracování.

Všimni si, že neúspěšné nebo zrušené úlohy se za určitých okolností počítají do tvého využití – podrobnosti najdeš v části [Neúspěšné a zrušené úlohy](#failed-job).

U uživatelů placených plánů určuje využití cenu pracovní zátěže. Podrobnosti najdeš v části [Správa nákladů](/guides/manage-cost).

<span id="failed-job"></span>
## Využití pro neúspěšné a zrušené úlohy
Pokud úloha selže nebo je zrušena, nahlášené využití je následující:

* Režim úlohy nebo dávky: Nahlášené využití je čas, po který byla QPU uzamčena pro vykonávání tvé pracovní zátěže, až do okamžiku selhání nebo zrušení. Pokud tedy k selhání nebo zrušení došlo před uzamčením, nahlášené využití je nula. V opačném případě je nahlášené využití pracovní zátěže rovno množství využití před jejím selháním nebo zrušením. Některé neúspěšné úlohy se tedy v nahlášeném využití neobjeví a jiné ano.

* Režim Session: Nahlášené využití je wall-clock čas, po který je session aktivní, bez ohledu na počet úloh, které selžou nebo jsou zrušeny.

<span id="view-usage"></span>
## Dotaz na skutečné využití pracovní zátěže
Po dokončení pracovní zátěže existuje několik způsobů, jak zobrazit její skutečné využití:

- Spusť [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) nebo [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) v `qiskit-ibm-runtime` 0.30 nebo novějším. Pokud používáš starší verzi `qiskit-ibm-runtime` (>= 0.23 a < 0.30), využití lze stále najít v `session.details()["usage_time"]` a `batch.details()["usage_time"]`.
- Použij [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) pro zobrazení využití konkrétní dávky nebo session.
- Použij [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) pro zobrazení využití jedné úlohy.

<span id="instance-usage"></span>
## Zobrazení využití instance
Využití instance si můžeš prohlédnout na stránce [Instances](https://quantum.cloud.ibm.com/instances), nebo na stránce [Analytics](https://quantum.cloud.ibm.com/analytics) (pokud máš příslušná oprávnění). Všimni si, že stránky mohou zobrazovat různé hodnoty využití, protože využití počítají odlišně.

Stránka Instances zobrazuje využití v reálném čase za posledních 28 dní (klouzavé okno), až do aktuálního času v aktuálním dni. Využití na stránce Analytics se přepočítává každou hodinu a zahrnuje posledních 28 celých dní; to znamená, že zobrazuje využití od 00:00 před 28 dny do dnešního dne, k celé hodině.

## Odhad využití před odesláním úlohy
Přestože přesný lokální odhad komplikují dodatečné operace prováděné pro potlačení a zmírňování chyb, můžeš použít tento základní vzorec pro přibližný odhad využití:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>` je režie přibližně 2 s na dílčí úlohu. Zahrnuje operace jako načítání dat do řídicí elektroniky. Tvá primitivní úloha může být rozdělena do více dílčích úloh, pokud je příliš velká na to, aby ji modul pro spouštění zpracoval najednou.
- `rep_delay` je [uživatelsky konfigurovatelná](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay) volba a její výchozí hodnota je dána `backend.default_rep_delay`, což je na většině backendů IBM Quantum 250 mikrosekund. Všimni si, že snížení `rep_delay` zkracuje celkovou dobu spuštění QPU, ale za cenu vyšší chybovosti při přípravě stavů; více informací najdeš v průvodci [Spuštění s dynamickou frekvencí opakování](/guides/repetition-rate-execution).
- `<circuit length>` je celková délka instrukce. Každá instrukce trvá na QPU jinou dobu, takže celková délka se liší circuit od circuit. Například měření může trvat 56krát déle než hradlo `x`. Pro zjištění přesné doby trvání každé instrukce lze použít `backend.target[<instruction>][<qubit>].duration`. Typická délka circuit se pravděpodobně pohybuje mezi 50–100 mikrosekundami. Pokud při práci s primitivy používáš techniky potlačení nebo zmírňování chyb, mohou být do tvého circuit vloženy další instrukce, čímž se celková délka circuit zvýší.
    > **Note:** [Experimentální volba `scheduler_timing`](/guides/visualize-circuit-timing) vrací celkový čas circuit, ale tento čas se NEPOUŽÍVÁ pro účtování.
- `<num executions>` je celkový počet circuits vynásobený počtem shotů, přičemž circuits jsou ty, které vzniknou po rozesílání prvků PUB.
    - Pokud při práci s primitivy používáš techniky zmírňování chyb, mohou být jako součást procesu zmírňování spuštěny další circuits, což zvýší celkový počet spuštění. Navíc pokročilé techniky zmírňování chyb jako PEA a PEC jsou spojeny s mnohem vyšší režií, protože vyžadují spuštění circuits pro učení se šumu.
    - Estimator seskupuje qubit-wise komutující observables, čímž snižuje počet spuštění.

Pokud nepoužíváš žádné pokročilé techniky zmírňování chyb ani vlastní `rep_delay`, můžeš jako rychlý vzorec použít `2+0.00035*<num executions>`.

## Další kroky
> **Tip:** - Přečti si tyto tipy: [Minimalizace doby spuštění úlohy](minimize-time).
>     - Nastav [Maximální dobu spuštění](max-execution-time).
>     - Nauč se transpilovat lokálně v sekci [Transpiler](./transpile/).
>     - Vyzkoušej průvodce [Porovnání nastavení transpilátoru](/guides/circuit-transpilation-settings).